# 01a — Groundsource: Urban Areas Detection

**Purpose:** Intersect each flood polygon with the GHS-BUILT (Global Human Settlements Built-up
Surface) raster to quantify the built-up area contained within it. This produces an `urban_percentage`
column that is used in all downstream steps to filter for genuinely urban flood events.

**Method:** Zonal statistics (sum of GHS-BUILT pixel values inside each polygon) are computed in
Mollweide equal-area projection (ESRI:54009), matching the native CRS of the raster. Processing
is parallelised across multiple CPU cores using `concurrent.futures.ProcessPoolExecutor`.

**Input:**
- `groundsource_2026.parquet` — raw flood events (geometry as WKB bytes, WGS84)
- `GHS_BUILT_S_E2020_GLOBE_R2023A_54009_100_V1_0.tif` — GHS-BUILT raster (100 m, Mollweide)

**Output:**
- `outputs/groundsource_urban_df.parquet` — original columns plus:
  - `urban_built_up_area_m2` : sum of GHS-BUILT pixel values within polygon (m²)
  - `polygon_total_area_m2`  : polygon area in Mollweide projection (m²)
  - `urban_percentage`       : built-up fraction of polygon area (%)

In [1]:
import os
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings('ignore')

In [2]:
# --- CONFIGURATION ---

INPUT_PATH    = r"D:\Development\RESEARCH\urban_flood_database\Groundsource\groundsource_2026.parquet"
GHS_RASTER    = r"D:\Development\RESEARCH\global_datasets\GHS_BUILT\GHS_BUILT_S_E2020_GLOBE_R2023A_54009_100_V1_0.tif"
OUTPUT_DIR    = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs"
OUTPUT_PATH   = os.path.join(OUTPUT_DIR, "groundsource_urban_df.parquet")

MOLLWEIDE_CRS = "ESRI:54009"   # native CRS of GHS-BUILT raster; equal-area
WGS84_CRS     = "EPSG:4326"    # CRS of the raw Groundsource geometry
N_WORKERS     = 6              # parallel CPU workers (reduce if RAM is tight)
CHUNK_SIZE    = 5_000          # rows per worker chunk

## 1. Load data and reproject to Mollweide

In [3]:
t_start = time.time()

df = pd.read_parquet(INPUT_PATH)
print(f"Loaded {len(df):,} records in {time.time() - t_start:.1f}s")

# Decode WKB bytes → shapely geometries in WGS84, then reproject to Mollweide
t1 = time.time()
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry'], crs=WGS84_CRS))
gdf = gdf.to_crs(MOLLWEIDE_CRS)
print(f"Decoded and reprojected {len(gdf):,} geometries in {time.time() - t1:.1f}s")

# Compute polygon area in Mollweide (accurate equal-area measurement)
gdf['polygon_total_area_m2'] = gdf.geometry.area
print(f"Area range: {gdf['polygon_total_area_m2'].min():.0f} – {gdf['polygon_total_area_m2'].max():.0f} m²")

Loaded 2,646,302 records in 3.8s
Decoded and reprojected 2,646,302 geometries in 24.4s
Area range: 1 – 5027064592 m²


## 2. Parallel zonal statistics

In [4]:
def _zonal_stats_worker(args):
    """
    Compute the sum of GHS-BUILT raster pixels within each polygon for one chunk.

    This function runs in a separate process. Geometries are passed as WKB bytes
    (serialisable) and reconstructed inside the worker.

    Args:
        args: tuple of (wkb_bytes_list, raster_path, mollweide_crs)

    Returns:
        list of float — urban built-up area sum in m² for each polygon in the chunk
    """
    wkb_bytes_list, raster_path, mollweide_crs = args
    import geopandas as gpd
    from rasterstats import zonal_stats

    geoms = gpd.GeoSeries.from_wkb(wkb_bytes_list, crs=mollweide_crs)
    results = zonal_stats(
        list(geoms),
        raster_path,
        stats=['sum'],
        all_touched=True,
        nodata=0
    )
    return [r.get('sum', 0) or 0 for r in results]

In [ ]:
CHUNK_DIR = os.path.join(OUTPUT_DIR, "urban_chunks")
os.makedirs(CHUNK_DIR, exist_ok=True)

wkb_mollweide = gdf.geometry.to_wkb()
total_chunks  = (len(gdf) + CHUNK_SIZE - 1) // CHUNK_SIZE

# Determine which chunks still need processing
done_chunks = {
    int(f.replace("chunk_", "").replace(".parquet", ""))
    for f in os.listdir(CHUNK_DIR)
    if f.startswith("chunk_") and f.endswith(".parquet")
}
pending = [i for i in range(total_chunks) if i not in done_chunks]

print(f"Total chunks : {total_chunks}")
print(f"Already done : {len(done_chunks)}")
print(f"Remaining    : {len(pending)}")

if pending:
    tasks = {
        idx: (wkb_mollweide.iloc[idx * CHUNK_SIZE : (idx + 1) * CHUNK_SIZE].tolist(),
              GHS_RASTER, MOLLWEIDE_CRS)
        for idx in pending
    }

    t2 = time.time()
    completed_count = len(done_chunks)

    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(_zonal_stats_worker, task): idx
                   for idx, task in tasks.items()}
        for future in as_completed(futures):
            idx    = futures[future]
            result = future.result()
            i_start = idx * CHUNK_SIZE

            # Save chunk immediately — survives a crash or kernel restart
            pd.DataFrame({
                'row_idx'               : list(range(i_start, i_start + len(result))),
                'urban_built_up_area_m2': result,
            }).to_parquet(os.path.join(CHUNK_DIR, f"chunk_{idx:04d}.parquet"), index=False)

            completed_count += 1
            elapsed = time.time() - t2
            pct  = completed_count / total_chunks * 100
            rate = (completed_count - len(done_chunks)) / elapsed * CHUNK_SIZE if elapsed > 0 else 0
            print(f"  chunk {idx:04d} saved | {completed_count}/{total_chunks} ({pct:.1f}%) | "
                  f"{elapsed:.0f}s elapsed | ~{rate:.0f} rows/s")

    print(f"\nAll pending chunks done in {time.time() - t2:.1f}s")
else:
    print("All chunks already complete — loading from disk.")

# Assemble results in row order from checkpoint files
urban_areas_by_chunk = []
for chunk_idx in range(total_chunks):
    df_chunk = pd.read_parquet(os.path.join(CHUNK_DIR, f"chunk_{chunk_idx:04d}.parquet"))
    urban_areas_by_chunk.append(df_chunk.sort_values('row_idx')['urban_built_up_area_m2'].tolist())

print(f"Loaded {sum(len(c) for c in urban_areas_by_chunk):,} results from {total_chunks} chunks")

## 3. Calculate urban percentage and save

In [ ]:
# Flatten results and attach to GeoDataFrame
all_urban_areas = [val for chunk_result in urban_areas_by_chunk for val in chunk_result]
gdf['urban_built_up_area_m2'] = all_urban_areas

# Urban percentage: built-up area as a fraction of total polygon area
gdf['urban_percentage'] = np.where(
    gdf['polygon_total_area_m2'] > 0,
    (gdf['urban_built_up_area_m2'] / gdf['polygon_total_area_m2']) * 100,
    0.0
)
# Clip to [0, 100] to absorb any floating-point rounding
gdf['urban_percentage'] = gdf['urban_percentage'].clip(0, 100)

print("Urban percentage statistics:")
print(gdf['urban_percentage'].describe())
print(f"\nEvents with urban_percentage > 0 : {(gdf['urban_percentage'] > 0).sum():,}")

In [ ]:
# Build output DataFrame: preserve original WKB geometry column, add new numeric columns
output_df = df.copy()
output_df['polygon_total_area_m2']  = gdf['polygon_total_area_m2'].values
output_df['urban_built_up_area_m2'] = gdf['urban_built_up_area_m2'].values
output_df['urban_percentage']       = gdf['urban_percentage'].values

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_df.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved {len(output_df):,} rows → {OUTPUT_PATH}")
print(f"Total runtime: {time.time() - t_start:.1f}s")